# 1) Lojistik Regresyon Nedir?
Lojistik regresyon aslında isminde regresyon geçmesine rağmen bir **sınıflandırma** algoritmasıdır. Lojistik Regresyon kullandığı sigmoid fonksiyonu yardımıyla lineer regresyonun ürettiği -sonsuz ila +sonsuz arasındaki çıktıyı 0-1 arasında olasılık değerine döndürür. Ayrıca, Lojistik regresyon Maksimum Olabilirlik (MLE) metodunu kullanır. 

### Sigmoid Fonksiyon Formülü
Sigmoid(z) = 1 / (1 + e^(-z))

Bu fonksiyon, her türlü sayıyı (ne kadar büyük/küçük olursa olsun) 0 ile 1 arasına sıkıştırır:

- z çok büyükse (örneğin +10) → Sigmoid(z) ≈ 1
- z çok küçükse (örneğin -10) → Sigmoid(z) ≈ 0
- z = 0 ise → Sigmoid(z) = 0.5 (tam ortada, kararsız nokta)

### Karar Verme
- Eğer Sigmoid(z) ≥ 0.5 → Sınıf 1 (Evet, spam vb, hasta..)
- Eğer Sigmoid(z) < 0.5 → Sınıf 0 (Hayır, spam değil, hasta değil..)

In [1]:
import seaborn as sns
titanic = sns.load_dataset('titanic')
titanic.info()
titanic.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [5]:
titanic = titanic.dropna(subset=['embark_town'])
titanic = titanic.drop(columns='deck')
titanic['age'] = titanic.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)
titanic.isna().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [9]:
# alive sütunu survived sütunu ile aynı bilgiyi taşıdığı için bu sütunu veri setinden çıkarıyoruz. Ek olarak embark_town'un kısaltması olan embarked'ı da veri setinden çıkarıyoruz.
import pandas as pd
titanic = titanic.drop(columns=['alive', 'embarked'])
titanic = pd.get_dummies(titanic, columns=['embark_town', 'sex'], drop_first=True)
titanic['adult_male'] = titanic['adult_male'].astype(int)
sinif_siralamasi = {
    'First':1,
    'Second':2,
    'Third':3
}
titanic['class'] = titanic['class'].map(sinif_siralamasi)
titanic.head()


,survived,pclass,age,sibsp,parch,fare,class,who,adult_male,alone,embark_town_Queenstown,embark_town_Southampton,sex_male
0,0,3,22.0,1,0,7.2500,3,man,1,False,False,True,True
1,1,1,38.0,1,0,71.2833,1,woman,0,False,False,False,False
2,1,3,26.0,0,0,7.9250,3,woman,0,True,False,True,False
3,1,1,35.0,1,0,53.1000,1,woman,0,False,False,True,False
4,0,3,35.0,0,0,8.0500,3,man,1,True,False,True,True


In [11]:
titanic['alone'] = titanic['alone'].astype(int)
titanic.head()

,survived,pclass,age,sibsp,parch,fare,adult_male,alone,embark_town_Queenstown,embark_town_Southampton,sex_male
0,0,3,22.0,1,0,7.2500,1,0,False,True,True
1,1,1,38.0,1,0,71.2833,0,0,False,False,False
2,1,3,26.0,0,0,7.9250,0,1,False,True,False
3,1,1,35.0,1,0,53.1000,0,0,False,True,False
4,0,3,35.0,0,0,8.0500,1,1,False,True,True


In [12]:
# Son olarak bool sütunları integer'a dönüştürelim
bool_sutunlar = titanic.select_dtypes(include='bool').columns
titanic[bool_sutunlar] = titanic[bool_sutunlar].astype(int)
titanic.head() # Son kontrol

,survived,pclass,age,sibsp,parch,fare,adult_male,alone,embark_town_Queenstown,embark_town_Southampton,sex_male
0,0,3,22.0,1,0,7.2500,1,0,0,1,1
1,1,1,38.0,1,0,71.2833,0,0,0,0,0
2,1,3,26.0,0,0,7.9250,0,1,0,1,0
3,1,1,35.0,1,0,53.1000,0,0,0,1,0
4,0,3,35.0,0,0,8.0500,1,1,0,1,1


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = titanic.drop(columns=['survived'])
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8258

Confusion Matrix:
[[92 17]
 [14 55]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.86       109
           1       0.76      0.80      0.78        69

    accuracy                           0.83       178
   macro avg       0.82      0.82      0.82       178
weighted avg       0.83      0.83      0.83       178



### Temel Metrik ve Formülleri

1. Accuracy (Doğruluk)
Accuracy = (TP + TN) / (TP + TN + FP + FN)

Ne anlatır: Modelin toplamda ne kadar doğru tahmin yaptığı (tüm sınıfları birlikte).
Ne zaman yanıltıcı olur: Sınıflar dengesiz olduğunda (örneğin %95 "hayır", %5 "evet" ise, hep "hayır" diyen bir model bile %95 accuracy alır ama işe yaramaz).

2. Precision (Kesinlik)
Precision = TP / (TP + FP)

Ne anlatır: Model "Pozitif" dediğinde, ne kadar güvenilir? Yani pozitif dediklerinin yüzde kaçı gerçekten pozitifmiş.
Ne zaman önemli: Yanlış alarmın (FP) maliyetli olduğu durumlarda örneğin spam filtresi (gerçek bir maili spam'e atmak kötü, precision önemli).

3. Recall (Duyarlılık / Sensitivity)
Recall = TP / (TP + FN)

Ne anlatır: Gerçekte pozitif olanların ne kadarını yakalayabildik?
Ne zaman önemli: Kaçırmanın (FN) maliyetli olduğu durumlarda örneğin kanser teşhisi (hasta birini "sağlıklı" demek çok tehlikeli, recall önemli).

4. F1-Score
F1 = 2 × (Precision × Recall) / (Precision + Recall)

Ne anlatır: Precision ve Recall'ın dengeli ortalaması (harmonik ortalama). İkisi arasında denge kurmak istediğinde kullanılır — biri çok yüksek diğeri çok düşükse F1 de düşük çıkar (harmonik ortalama, bu dengesizliği cezalandırır).

Basit Bir Akılda Tutma Yöntemi
- Precision: "Dediğim şeylerin kaçı doğru?" (model iddialarının güvenilirliği)
- Recall: "Olması gerekenlerin kaçını yakaladım?" (kaçırma oranı)
- Trade-off var: Genelde birini artırmaya çalışırken diğeri düşer (eşik değerini değiştirerek bunu kontrol edebilirsin — 0.5 yerine 0.3 gibi düşük bir eşik seçersen Recall artar ama Precision düşer).

Logistic Regression'ın Varsayımları

1. Bağımlı Değişken Kategorik Olmalı
y, iki (binary) veya daha fazla kategoriden oluşmalı (bizim örnekte 0/1).

2. Gözlemler Birbirinden Bağımsız Olmalı
Linear Regression'daki ile aynı — bir gözlem diğerini etkilememeli.

3. Bağımsız Değişkenler Arasında Multicollinearity Olmamalı
Yine Faz 2'den bildiğin kavram — XᵀX matrisinin kararlılığı için gerekli (bizim pclass/class çakışmasını bu yüzden temizlemiştik).

4. Logit ile Bağımsız Değişkenler Arasında Doğrusal İlişki
İlginç bir nokta: Logistic Regression, y ile x arasında doğrusal ilişki aramaz (zaten sigmoid eğrisel), ama log-odds (logit) ile x arasında doğrusal bir ilişki bekler:

logit(p) = ln(p/(1-p)) = b + w×x

5. Büyük Örneklem Boyutu Gerektirir
Linear Regression'dan farklı olarak, Logistic Regression'ın katsayıları MLE (Maximum Likelihood Estimation) ile bulunuyor — bu yöntem, küçük veri setlerinde kararsız/güvenilmez sonuçlar verebilir, bu yüzden yeterince büyük bir örneklem gerekir.

### Logistic Regression Sonuçları

Model, Titanic veri setinde %82.58 accuracy elde etti.

Confusion Matrix:
- TN=92, FP=17, FN=14, TP=55

Classification Report:
- Sınıf 0 (hayatta kalmayan): Precision=0.87, Recall=0.84
- Sınıf 1 (hayatta kalan): Precision=0.76, Recall=0.80

Model, Sınıf 0'ı Sınıf 1'e göre biraz daha iyi tahmin ediyor — bunun olası bir sebebi, Sınıf 0'ın veri setinde daha fazla örneğe sahip olması (support=109 vs 69), yani model bu sınıfı öğrenmek için daha fazla veri görmüş olabilir.